# Brain tumor MRI: model comparison on a free Colab GPU

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`.
2. Upload `processed.zip` (made on your laptop, see `docs/COLAB.md`) to Google Drive at `My Drive/btmri/processed.zip`.

Then `Runtime -> Run all`. All results are saved to `My Drive/btmri/runs`.

**If Colab restarts or disconnects** (it forgets everything when that happens): run the **setup cell** again, then continue from where you were. Re-running is always safe: models that already finished are skipped.

In [ ]:
# SETUP: safe to re-run at any time. Re-run it after every restart or disconnect.
import glob
import os

from google.colab import drive

DRIVE = "/content/drive/MyDrive/btmri"
RUNS = f"{DRIVE}/runs"
REPO = "https://github.com/sallew-glitch/brain-tumor-mri-gradcam.git"

drive.mount("/content/drive")  # a pop-up asks for permission the first time

if not os.path.exists("/content/project"):
    !git clone -q {REPO} /content/project
%cd /content/project
!git pull -q
!pip install -q -e ".[dev]"

if not os.path.exists("data/processed/metadata.csv"):
    !mkdir -p data && unzip -q -o {DRIVE}/processed.zip -d data/
print("images:", len(os.listdir("data/processed/images")), "(expected 3064)")

!nvidia-smi --query-gpu=name --format=csv,noheader
!python -m pytest -q

In [ ]:
# TRAIN every model on fold 1 (roughly 20-40 minutes in total on a T4).
# Models that already have a finished fold-1 run in Drive are skipped.
# resnet18_cpu is the laptop baseline config; it runs fine on a GPU too.
MODELS = ["resnet18_cpu", "efficientnet_b0", "densenet121", "resnet50"]

for model in MODELS:
    if glob.glob(f"{RUNS}/{model}_*/fold_1/test_metrics.json"):
        print(f"{model}: already trained, skipping")
        continue
    !python scripts/train.py --config configs/{model}.yaml --folds 1 --set output_dir={RUNS}

In [ ]:
# GRAD-CAM on the VALIDATION fold at both layers, for every finished run.
for run in sorted(glob.glob(f"{RUNS}/*/summary.json")):
    run_dir = os.path.dirname(run)
    !python scripts/explain.py --run-dir "{run_dir}" --split val --layers last penultimate

In [ ]:
# COMPARE. Choose using the VAL_macro_f1 column and the Grad-CAM table (val split).
!python scripts/compare_runs.py --runs-dir {RUNS}

## After choosing a model

Run the full 5-fold cross-validation for the chosen model (edit the name), then the final Grad-CAM report on the test folds at the layer you picked:

```python
!python scripts/train.py --config configs/efficientnet_b0.yaml --set output_dir={RUNS}
!python scripts/explain.py --run-dir "{RUNS}/<new_run_name>" --split test --layers penultimate
```

To look at a heatmap image here:

```python
from IPython.display import Image
Image(f"{RUNS}/<run_name>/fold_1/gradcam_val_last/examples.png")
```